In [1]:
import os, time, gc
import numpy as np
import pandas as pd
import torch

import torchvision.models as models

from datasets import load_dataset, load_from_disk
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

import warnings
# Hide the specific PyTorch positional args warning
warnings.filterwarnings("ignore", category=UserWarning, message="positional args are being deprecated")


AI_ACCEL = "IPU" # or "GPU"

import poptorch

## Dataset / transform / adapter

In [2]:
class Camelyon17Dataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None, return_metadata=False):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.return_metadata = return_metadata

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        sample = self.hf_dataset[idx]
        image = sample["image"].convert("RGB")
        label = torch.tensor(sample["label"], dtype=torch.long)

        if self.transform is not None:
            image = self.transform(image)

        if self.return_metadata:
            metadata = {k: sample[k] for k in [
                "center", "image_id", "patient", "node",
                "x_coord", "y_coord", "slide"
            ]}
            return image, label, metadata

        return image, label


my_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])


def load_camelyon17():
    my_dataset = "wltjr1007/Camelyon17-WILDS"
    path = "/home/work/datasets/camelyon17_wilds"
    train_path = "/home/work/datasets/camelyon17_wilds/train"

    if os.path.exists(path) and os.path.exists(train_path) and os.listdir(train_path):
        print("Loading dataset from local disk...")
        return load_from_disk(path)

    print("Downloading dataset...")
    ds = load_dataset(my_dataset, cache_dir=path)
    ds.save_to_disk(path)
    return ds


def make_random_subset(hf_dataset, n_samples=None, seed=42):
    shuffled = hf_dataset.shuffle(seed=seed)
    if n_samples is None:
        return shuffled
    return shuffled.select(range(min(n_samples, len(shuffled))))


def make_balanced_subset(hf_dataset, n_per_class=5000, seed=42, label_key="label"):
    shuffled = hf_dataset.shuffle(seed=seed)
    selected, indices = {}, []

    for i, label in enumerate(shuffled[label_key]):
        label = int(label)
        selected.setdefault(label, 0)

        if selected[label] < n_per_class:
            indices.append(i)
            selected[label] += 1

        if len(selected) >= 2 and all(v >= n_per_class for v in selected.values()):
            break

    return shuffled.select(indices)

##  Model factory

In [3]:
class ResNetCamelyon(torch.nn.Module):
    def __init__(self, model_type="ResNet18", num_classes=2, pretrained=True):
        super().__init__()

        if model_type == "ResNet18":
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet18(weights=weights)

        elif model_type == "ResNet34":
            weights = models.ResNet34_Weights.DEFAULT if pretrained else None
            self.backbone = models.resnet34(weights=weights)

        elif model_type == "ResNet50":
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            self.backbone = models.resnet50(weights=weights)

        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        in_features = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(in_features, num_classes)

    def forward(self, x, labels=None):
        logits = self.backbone(x)

        if labels is None:
            return logits

        loss = torch.nn.functional.cross_entropy(logits, labels)
        return logits, loss

## Cleanup - wrappers

In [4]:
def cleanup_poptorch():
    for name in ["poptorch_train", "poptorch_val", "infer_model", "display_infer_model"]:
        obj = globals().get(name, None)
        if obj is None:
            continue

        try:
            obj.detachFromDevice()
            print(f"{name}: detached")
        except Exception as e:
            print(f"{name}: detach skipped:", e)

        try:
            obj.destroy()
            print(f"{name}: destroyed")
        except Exception as e:
            print(f"{name}: destroy skipped:", e)

    gc.collect()

## Evaluation 

In [5]:
#_____________________
# IPU based functions
#_____________________

def evaluate_ipu(infer_model, data_loader):
    infer_model.eval()
    total, correct = 0, 0
    total_loss, n_steps = 0.0, 0

    with torch.no_grad():
        for images, labels in data_loader:
            logits = infer_model(images)
            #if n_steps ==0:
            #    print(logits.shape)
            loss = torch.nn.functional.cross_entropy(logits, labels)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)
            total_loss += loss.item()
            n_steps += 1
            

    return total_loss / n_steps, correct / total


def test_ipu(model_type, checkpoint, test_hf, transform, infer_opts):
    test_model = ResNetCamelyon(model_type=model_type, pretrained=False)
    test_model.load_state_dict(torch.load(checkpoint))
    test_model.eval()

    test_data = Camelyon17Dataset(test_hf, transform=transform)

    test_loader = poptorch.DataLoader(
        infer_opts,
        test_data,
        batch_size=10,
        shuffle=False,
        drop_last=True,
        num_workers=4
    )

    infer_model = poptorch.inferenceModel(test_model, options=infer_opts)

    all_preds, all_labels, all_probs = [], [], []

    start = time.time()

    with torch.no_grad():
        for images, labels in test_loader:
            logits = infer_model(images)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs[:, 1].cpu().tolist())

    elapsed = time.time() - start
    n_eval = len(all_labels)
    
    from collections import Counter

    print(f"Preds : 0={Counter(all_preds)[0]}, 1={Counter(all_preds)[1]}")
    print(f"Labels: 0={Counter(all_labels)[0]}, 1={Counter(all_labels)[1]}")

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()

    metrics = {
        "test_accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="weighted"),
        "recall": recall_score(all_labels, all_preds, average="weighted"),
        "f1": f1_score(all_labels, all_preds, average="weighted"),
        "auc": roc_auc_score(all_labels, all_probs),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "sensitivity": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "test_images_eval": n_eval,
        "inference_sec": elapsed,
        "images_per_sec": n_eval / elapsed,
        "ms_per_image": elapsed / n_eval * 1000,
    }

    try:
        infer_model.detachFromDevice()
        infer_model.destroy()
    except Exception:
        pass

    return metrics

## Experiment Function

In [6]:
from transformers.optimization import Adafactor 

def freeze_batchnorm(model):
    for m in model.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False
                
def run_experiment(config, dataset):
    cleanup_poptorch()

    model_type = config["model_type"]
    save_file = config["save_file"]
    
    #print("model type:", model_type)
    #print("save file at the beginning:", save_file) 

    # Data split
    if config["whole_data"]:
        train_hf = dataset["train"]
        val_hf = dataset["validation"]
        test_hf = dataset["test"]
    else:
        if config["balanced"]:
            train_hf = make_balanced_subset(dataset["train"], config["n_train_per_class"], seed=42)
            val_hf = make_balanced_subset(dataset["validation"], config["n_val_per_class"], seed=43)
            test_hf = make_balanced_subset(dataset["test"], config["n_test_per_class"], seed=44)
        else:
            train_hf = make_random_subset(dataset["train"], config["n_train"], seed=42)
            val_hf = make_random_subset(dataset["validation"], config["n_val"], seed=43)
            test_hf = make_random_subset(dataset["test"], config["n_test"], seed=44)

    # save_file defined here
    save_file  = (
                f"./models/{config['model_type']}_"
                f"train{len(train_hf)}_"
                f"bat{config['batch_size']}_"
                f"grad{config['grad_accum']}_"
                f"lr{config['lr']}_"
                f"best.pt"
            )
    
    #print("\nSave file initiated : ", save_file)
    
    #Options
    
    train_opts = poptorch.Options()
    train_opts.TensorLocations.setOptimizerLocation(
        poptorch.TensorLocationSettings().useOnChipStorage(False)
    )
    train_opts.setAvailableMemoryProportion({f"IPU{i}": 0.6 for i in range(8)})
    train_opts.replicationFactor(config["rep_fact"])
    train_opts.deviceIterations(config["device_itr"])
    train_opts.Training.gradientAccumulation(config["grad_accum"])
    train_opts.Precision.setPartialsType(torch.float16)
    #train_opts.enableExecutableCaching("./my_cache")

    infer_opts = poptorch.Options()
    infer_opts.setAvailableMemoryProportion({f"IPU{i}": 0.6 for i in range(8)})
    infer_opts.replicationFactor(config["rep_fact"])
    infer_opts.deviceIterations(config["device_itr"])
    infer_opts.Precision.setPartialsType(torch.float16)
    #infer_opts.enableExecutableCaching("./my_cache")


    # Data assignment
    train_data = Camelyon17Dataset(train_hf, transform=my_transform)
    val_data = Camelyon17Dataset(val_hf, transform=my_transform)

    # Data loader
    train_loader = poptorch.DataLoader(
        train_opts,
        train_data,
        batch_size=config["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=4
    )

    #print("/nInfer opts given for val_loader:", infer_opts)
    val_loader = poptorch.DataLoader(
        infer_opts,
        val_data,
        batch_size=10,
        shuffle=False,
        drop_last=True,
        num_workers=4
    )

    # Model instanciation
    train_model = ResNetCamelyon(model_type=model_type, pretrained=True)
    
    val_model = ResNetCamelyon(model_type=model_type, pretrained=False)
    val_model.load_state_dict(train_model.state_dict())
    val_model.eval()

    optimizer = torch.optim.AdamW(
        train_model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    # Model wrappers
    poptorch_train = poptorch.trainingModel(
        train_model,
        options=train_opts,
        optimizer=optimizer
    )

    poptorch_val = poptorch.inferenceModel(
        val_model,
        options=infer_opts
    )

    val_images0, _ = next(iter(val_loader))
    with torch.no_grad():
        _ = poptorch_val(val_images0)

    best_val_loss = float("inf")
    best_val_acc = 0.0
    current_lr = config["lr"]

    epochs_without_improvement = 0
    epochs_without_lr_improvement = 0

    start_train = time.time()

    for epoch in range(config["num_epochs"]):
        t0 = time.time()
        running_loss, n_steps = 0.0, 0

        poptorch_train.train()

        for images, labels in train_loader:
            logits, loss = poptorch_train(images, labels)
            running_loss += loss.mean().item()
            n_steps += 1

        train_loss = running_loss / n_steps

        poptorch_train.copyWeightsToHost()
        val_model.load_state_dict(train_model.state_dict())
        poptorch_val.copyWeightsToDevice()

        val_loss, val_acc = evaluate_ipu(poptorch_val, val_loader) #poptorch_val will get .eval() inside

        print(
            f"\n{model_type} | epoch {epoch+1}/{config['num_epochs']} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | lr={current_lr:.2e} | "
            f"time={time.time()-t0:.1f}s"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            epochs_without_improvement = 0
            epochs_without_lr_improvement = 0

            poptorch_train.copyWeightsToHost()
            torch.save(train_model.state_dict(), save_file)

            print("✅Saved best model:", save_file, "val_loss:", val_loss)  #✅
            
        else:
            epochs_without_improvement += 1
            epochs_without_lr_improvement += 1

        if epochs_without_lr_improvement >= config["lr_patience"]:
            new_lr = max(current_lr * config["lr_factor"], config["min_lr"])

            if new_lr < current_lr:
                current_lr = new_lr
                print(f"Reducing LR to {current_lr:.2e}")

                for group in optimizer.param_groups:
                    group["lr"] = current_lr

                poptorch_train.setOptimizer(optimizer)
                epochs_without_lr_improvement = 0

        if epochs_without_improvement >= config["patience"]:
            print("Early stopping triggered.")
            break

    train_time_min = (time.time() - start_train) / 60

    poptorch_train.copyWeightsToHost()

    try:
        poptorch_train.detachFromDevice()
        poptorch_train.destroy()
        poptorch_val.detachFromDevice()
        poptorch_val.destroy()
    except Exception as e:
        print("cleanup warning:", e)

    #print("\nModel given to test_ipu", save_file)
    #print("\nInfer_opts given to test_ipu", infer_opts)
    test_metrics = test_ipu(
        model_type=model_type,
        checkpoint=save_file,
        test_hf=test_hf,
        transform=my_transform,
        infer_opts=infer_opts
    )
    
    print("Test accuracy =", test_metrics["test_accuracy"])
          
    result = {
        "model_type": model_type,
        "whole_data": config["whole_data"],
        "balanced": config["balanced"],
        "train_size": len(train_hf),
        "val_size": len(val_hf),
        "test_size": len(test_hf),
        "batch_size": config["batch_size"],
        "rep_fact": config["rep_fact"],
        "device_itr": config["device_itr"],
        "grad_accum": config["grad_accum"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "best_val_loss": best_val_loss,
        "best_val_acc": best_val_acc,
        "train_time_min": train_time_min,
        "checkpoint": save_file,
        **test_metrics,
    }

    return result




## Preparation for multiple experiments 

In [7]:
WHOLE_DATA = False
LEARN_RATE = 1e-4
LEARN_FACTOR = 0.5
EPOCHS = 30
N_TRAIN = 50000
N_TEST = 5000
N_VALID = 5000
DEVICE_ITER = 1
REP_FACT = 4

BATCH_SIZE = 12
GRAD_ACCUM = 8

BATCH_SIZE_34 = 8
GRAD_ACCUM_34 = 8

BATCH_SIZE_50 = 2
GRAD_ACCUM_50 = 1

LR_PAT = 5
EARLY_PAT = 11

resnet18_config = {
        "model_type": "ResNet18",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": N_TRAIN,
        "n_val": N_VALID,
        "n_test": N_TEST,
        "batch_size": BATCH_SIZE,
        "rep_fact": REP_FACT,
        "device_itr": DEVICE_ITER,
        "grad_accum": GRAD_ACCUM,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs":EPOCHS,
        "patience": EARLY_PAT,
        "lr_patience": LR_PAT,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": "./models/best_resnet18_camelyon17_wilds.pt", #dummy
    }

resnet34_config = {
        "model_type": "ResNet34",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": N_TRAIN,
        "n_val": N_VALID,
        "n_test": N_TEST,
        "batch_size": BATCH_SIZE_34,
        "rep_fact": REP_FACT,
        "device_itr": DEVICE_ITER,
        "grad_accum": GRAD_ACCUM_34,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs": EPOCHS,
        "patience": EARLY_PAT,
        "lr_patience": LR_PAT,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": "./models/best_resnet34_camelyon17_wilds.pt", #dummy
    }

resnet50_config = {
        "model_type": "ResNet50",
        "whole_data": WHOLE_DATA,
        "balanced": False,
        "n_train": N_TRAIN,
        "n_val": N_VALID,
        "n_test": N_TEST,
        "batch_size": BATCH_SIZE_50,
        "rep_fact": REP_FACT,
        "device_itr": DEVICE_ITER,
        "grad_accum": GRAD_ACCUM_50,
        "lr": LEARN_RATE,
        "weight_decay": 1e-4,
        "num_epochs": EPOCHS,
        "patience": EARLY_PAT,
        "lr_patience": LR_PAT,
        "lr_factor": LEARN_FACTOR,
        "min_lr": 1e-7,
        "save_file": "./models/best_resnet50_camelyon17_wilds.pt",  #dummy
    }

## Setting configurations for multiple exps.

In [8]:
import copy
import itertools
import traceback
from datetime import datetime

def make_config_grid(base_configs, search_space):
    configs = []

    keys = list(search_space.keys())
    values = list(search_space.values())

    for base in base_configs:
        for combo in itertools.product(*values):
            cfg = copy.deepcopy(base)

            for k, v in zip(keys, combo):
                # If it's our paired tuple, unpack it explicitly
                if k == "dataset_splits":
                    cfg["n_train"], cfg["n_val"], cfg["n_test"] = v
                else:
                    cfg[k] = v

            cfg["run_id"] = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            cfg["save_file"] = (
                f"./models/{cfg['model_type']}_"
                f"train{cfg.get('n_train', 'full')}_"
                f"rep{cfg['rep_fact']}_"
                f"di{cfg['device_itr']}_"
                f"ga{cfg['grad_accum']}_"
                f"lr{cfg['lr']}_"
                f"{cfg['run_id']}_best.pt"
            )

            configs.append(cfg)

    return configs

## Running with multiple models and hyperparameters

In [ ]:
dataset = load_camelyon17()
full_train = len(dataset['train'])
full_test = len(dataset['test'])
full_valid = len(dataset['validation'])

os.makedirs("./models", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./my_cache", exist_ok=True)

base_configs  = [
    #resnet18_config,
    #resnet34_config,
    resnet50_config,
]


CSV_FILE = "./logs/camelyon17_ipu_benchmark.csv"
#ERROR_CSV_FILE = "./logs/camelyon17_ipu_errors.csv"

search_space = {
    "lr": [1e-4],
    #"batch_size": [12],
    #"grad_accum": [8],
    # Pair (n_train, n_val, n_test) together
    "dataset_splits": [
        #(full_train, full_valid, full_test), # Experiment 1 splits
        (50000, 5000, 5000)  # Experiment 2 splits
    ]
}

experiments = make_config_grid(base_configs, search_space)
print(f"Number of {AI_ACCEL} experiments:", len(experiments))

results = []

for i, config in enumerate(experiments):
    print("\n" + "=" * 90)
    print(f'{AI_ACCEL} Experiment {i+1}/{len(experiments)} : {config["model_type"]}')
    print(config)
    print("=" * 90)
            
    result = run_experiment(config, dataset)
            
    pd.DataFrame([result]).to_csv(
        CSV_FILE,
        mode="a",
        header=not os.path.exists(CSV_FILE),
        index=False
    )

    print("Logged:", CSV_FILE)

    cleanup_poptorch()
     


Loading dataset from local disk...
Number of IPU experiments: 1

IPU Experiment 1/1 : ResNet50
{'model_type': 'ResNet50', 'whole_data': False, 'balanced': False, 'n_train': 50000, 'n_val': 5000, 'n_test': 5000, 'batch_size': 2, 'rep_fact': 4, 'device_itr': 1, 'grad_accum': 1, 'lr': 0.0001, 'weight_decay': 0.0001, 'num_epochs': 30, 'patience': 11, 'lr_patience': 5, 'lr_factor': 0.5, 'min_lr': 1e-07, 'save_file': './models/ResNet50_train50000_rep4_di1_ga1_lr0.0001_20260701_171749_088196_best.pt', 'run_id': '20260701_171749_088196'}


Graph compilation: 100%|██████████| 100/100 [05:30<00:00]



ResNet50 | epoch 1/30 | train_loss=0.1700 | val_loss=0.4614 | val_acc=0.9342 | lr=1.00e-04 | time=1095.5s
✅Saved best model: ./models/ResNet50_train50000_bat2_grad1_lr0.0001_best.pt val_loss: 0.46138727441430094
